In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("qoute_dataset.csv")

In [3]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [4]:
df.shape

(3038, 2)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3038 entries, 0 to 3037
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   quote   3038 non-null   object
 1   Author  3038 non-null   object
dtypes: object(2)
memory usage: 47.6+ KB


In [6]:
quotes = df["quote"]
quotes.head()

0    “The world as we have created it is a process ...
1    “It is our choices, Harry, that show what we t...
2    “There are only two ways to live your life. On...
3    “The person, be it gentleman or lady, who has ...
4    “Imperfection is beauty, madness is genius and...
Name: quote, dtype: object

In [7]:
quotes = quotes.str.lower()

In [8]:
import string
translator = str.maketrans("","",string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))

In [9]:
from tensorflow.keras.preprocessing.text import Tokenizer

I0000 00:00:1784657387.599020   37658 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1784657387.599547   37658 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1784657387.646092   37658 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1784657388.834458   37658 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

In [10]:
vocab_size = 2000
tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(quotes)

In [11]:
word_index = tokenizer.word_index
print(len(word_index))
list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [12]:
sequence = tokenizer.texts_to_sequences(quotes)
print(quotes[0])
print(sequence[0])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70]


In [13]:
X = []
y = []

for seq in sequence:
    for i in range(1,len(seq)):
        input_seq = seq[:i]
        output_seq = seq[i]
        X.append(input_seq)
        y.append(output_seq)      

In [14]:
max_len = max(len(x) for x in X)
max_len

622

In [15]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [16]:
X_padded = pad_sequences(
    X,
    maxlen=max_len,
    padding='pre'
)

print(len(X_padded[0]))

622


In [17]:
y = np.array(y)
X_padded.shape

(76158, 622)

In [18]:
from tensorflow.keras.utils import to_categorical
y_onehot_encod = to_categorical(y,num_classes = vocab_size)
print(y_onehot_encod.shape)
print(y.shape)

(76158, 2000)
(76158,)


### above we done with tokenization and vectorization

In [19]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,SimpleRNN

In [20]:
embedding_dim = 50
rnn_units = 128

In [21]:
rnn_model = Sequential()
rnn_model.add(
    Embedding(input_dim=vocab_size,output_dim=embedding_dim,input_length=max_len,name = "embedding_layer")
)
rnn_model.add(
    SimpleRNN(units=rnn_units,name = "simpleRnn_layer")
)
rnn_model.add(
    Dense(units=vocab_size,activation="softmax",name = "output_layer")
)

E0000 00:00:1784657390.936391   37658 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [22]:
rnn_model.compile(
    optimizer='adam',
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

In [23]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_layer (Embedding)     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simpleRnn_layer (SimpleRNN)     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [24]:
lstm_model = Sequential()
lstm_model.add(
    Embedding(input_dim=vocab_size,output_dim=embedding_dim,input_length=max_len,name = "embedding_layer")
)
lstm_model.add(
    SimpleRNN(units=rnn_units,name = "simpleRnn_layer")
)
lstm_model.add(
    Dense(units=vocab_size,activation="softmax",name = "output_layer")
)

In [25]:
lstm_model.compile(
    optimizer='adam',
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

In [26]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_layer (Embedding)     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simpleRnn_layer (SimpleRNN)     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [28]:
epochs =5
batch_size = 128

In [29]:
history_rnn = rnn_model.fit(X_padded,y_onehot_encod,batch_size=batch_size,epochs=epochs,validation_split=0.1,verbose=1)

Epoch 1/5
536/536 ━━━━━━━━━━━━━━━━━━━━ 124s 229ms/step - accuracy: 0.0514 - loss: 5.9063 - val_accuracy: 0.0846 - val_loss: 5.6378
Epoch 2/5
536/536 ━━━━━━━━━━━━━━━━━━━━ 140s 262ms/step - accuracy: 0.0980 - loss: 5.4221 - val_accuracy: 0.1062 - val_loss: 5.3860
Epoch 3/5
536/536 ━━━━━━━━━━━━━━━━━━━━ 129s 240ms/step - accuracy: 0.1168 - loss: 5.1784 - val_accuracy: 0.1171 - val_loss: 5.2944
Epoch 4/5
536/536 ━━━━━━━━━━━━━━━━━━━━ 160s 299ms/step - accuracy: 0.1312 - loss: 5.0014 - val_accuracy: 0.1254 - val_loss: 5.2635
Epoch 5/5
536/536 ━━━━━━━━━━━━━━━━━━━━ 162s 303ms/step - accuracy: 0.1419 - loss: 4.8532 - val_accuracy: 0.1253 - val_loss: 5.2459


In [41]:
rnn_model.save("rnn_model.h5")

In [42]:
index_to_word = {}
for word, index in word_index.items():
  index_to_word[index] = word

In [43]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [44]:
def predictor(model,tokenizer,text,max_len):
  text = text.lower()

  seq = tokenizer.texts_to_sequences([text])[0]
  seq = pad_sequences([seq], maxlen=max_len, padding='pre')

  pred = model.predict(seq,verbose = 0)
  pred_index = np.argmax(pred)
  return index_to_word[pred_index]


In [48]:
seed_text = "It is our choices, "
next_word = predictor(rnn_model,tokenizer,seed_text,max_len)
print(next_word)

and


In [56]:
def generate_text(model,tokenizer,seed_text,max_len,n_words):
  for _ in range(n_words):
    next_word = predictor(model,tokenizer,seed_text,max_len)
    if next_word == "":
      break
    seed_text += " " + next_word
  return seed_text

In [57]:
seed = "we have created"
generate_text = generate_text(rnn_model,tokenizer,seed,max_len,10)
print(generate_text)

we have created the world is not the world is not a good
